In [1]:
!pip install gradio
!pip install mistralai --upgrade
!pip install mistralai gradio --quiet

     ---------------------------------------- 0.0/43.1 kB ? eta -:--:--
     ---------------------------------------- 43.1/43.1 kB ? eta 0:00:00
     ---------------------------------------- 0.0/90.6 kB ? eta -:--:--
     ---------------------------------------- 90.6/90.6 kB 2.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/42.9 MB ? eta -:--:--
   ---------------------------------------- 0.1/42.9 MB 2.2 MB/s eta 0:00:20
   ---------------------------------------- 0.3/42.9 MB 2.9 MB/s eta 0:00:15
   ---------------------------------------- 0.4/42.9 MB 2.9 MB/s eta 0:00:15
    --------------------------------------- 0.6/42.9 MB 2.9 MB/s eta 0:00:15
    --------------------------------------- 0.7/42.9 MB 2.8 MB/s eta 0:00:15
    --------------------------------------- 0.8/42.9 MB 2.9 MB/s eta 0:00:15
    --------------------------------------- 1.0/42.9 MB 3.0 MB/s eta 0:00:14
   - -------------------------------------- 1.2/42.9 MB 3.1 MB/s eta 0:00:14
   - ------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
anaconda-cloud-auth 0.1.4 requires pydantic<2.0, but you have pydantic 2.12.5 which is incompatible.


   ---------------------------------------- 0.0/509.3 kB ? eta -:--:--
   -------- ------------------------------- 112.6/509.3 kB 6.4 MB/s eta 0:00:01
   --------------------- ------------------ 276.5/509.3 kB 3.4 MB/s eta 0:00:01
   ------------------------------ --------- 389.1/509.3 kB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 509.3/509.3 kB 3.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/160.3 kB ? eta -:--:--
   ---------------------------------------- 160.3/160.3 kB 4.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/68.7 kB ? eta -:--:--
   ---------------------------------------- 68.7/68.7 kB 3.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/72.1 kB ? eta -:--:--
   ---------------------------------------- 72.1/72.1 kB 3.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/142.0 kB ? eta -:--:--
   ------------------------------------- -- 133.1/142.0 kB 4.0 MB/s eta 0:00:01
   --------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.75 requires requests_mock, which is not installed.
conda-repo-cli 1.0.75 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.75 requires PyYAML==6.0.1, but you have pyyaml 6.0.3 which is incompatible.
streamlit 1.30.0 requires protobuf<5,>=3.20, but you have protobuf 6.33.5 which is incompatible.
tensorflow-intel 2.15.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 6.33.5 which is incompatible.


In [2]:
import json
from dataclasses import dataclass
from typing import Literal, Optional, List

import time
import requests

import gradio as gr
from mistralai import Mistral

# 1) API CONFIG

In [ ]:
MISTRAL_API_KEY = "XXXXXXXXXXXXXXXXXXXXXXXXXXX"

MISTRAL_MODEL = "mistral-tiny"

client = Mistral(api_key=MISTRAL_API_KEY)

LAST_ERROR_MESSAGE = ""
PATIENT_REQUESTS: List["PatientRequest"] = []

GLADIA_API_KEY = "XXXXXXXXXXXXXXXXXXXXXXXXXXX"
GLADIA_BASE_URL = "https://api.gladia.io"

# 2) DATA CLASSES

In [4]:
UrgencyLevel = Literal["low", "medium", "high"]

@dataclass
class PreDiagnosis:
    condition: str
    urgencyLevel: UrgencyLevel
    symptoms: str

@dataclass
class PatientRequest:
    name: str
    symptoms: str
    temperature_c: float
    blood_pressure: str
    heart_rate_bpm: int
    pre_diagnosis: Optional[PreDiagnosis] = None

# 3) LLM PREDICTION LOGIC

In [5]:
SYSTEM_PROMPT = """
You are an assistant helping emergency medical dispatch (SAMU) with triage.
The user will provide a description of symptoms, and sometimes vital signs.

You will receive a JSON object with the following fields:
- "name": patient name (string)
- "symptoms": free-text symptoms description (string)
- "temperature_c": body temperature in °C (number)
- "blood_pressure": blood pressure as a string, (string)
- "heart_rate_bpm": heart rate in beats per minute (number)

Your job:
1. Infer the most likely condition or a short description (in plain language, not ICD codes).
2. Assign an urgency level: "low", "medium", or "high".
   - "high": life-threatening or potentially life-threatening, needs immediate attention
     (e.g. chest pain + shortness of breath, stroke signs, major trauma,
      severe breathing difficulty, altered consciousness, etc.)
   - "medium": needs relatively quick medical assessment (same day / a few hours)
     but not immediately life-threatening.
   - "low": non-urgent, could be handled by routine outpatient care or teleconsultation.

Use BOTH the symptoms and the vital signs. For example:
- Very high fever, very low blood pressure, very high heart rate → more urgent.
- Normal vitals and mild symptoms → less urgent.

Respond ONLY in valid JSON with the following fields:
{
  "condition": string,
  "urgencyLevel": "low" | "medium" | "high"
}

No extra text, no explanations, no markdown.
""".strip()

def _extract_json(text: str):
    try:
        return json.loads(text)
    except:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1:
        return json.loads(text[start:end+1])

    raise ValueError("Model did not return valid JSON: " + text)


def make_prediagnosis(symptoms: str) -> Optional[PreDiagnosis]:
    global LAST_ERROR_MESSAGE
    try:
        res = client.chat.complete(
            model=MISTRAL_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": symptoms},
            ],
            temperature=0.1
        )

        content = res.choices[0].message.content
        parsed = _extract_json(content)

        condition = parsed.get("condition", "Unknown condition")

        urgencyLevel_raw = parsed.get("urgencyLevel", "medium")
        urgencyLevel = str(urgencyLevel_raw).strip().lower()
        if urgencyLevel not in ("low", "medium", "high"):
            urgencyLevel = "medium"

        LAST_ERROR_MESSAGE = ""
        return PreDiagnosis(
            condition=condition,
            urgencyLevel=urgencyLevel,
            symptoms=symptoms,
        )

    except Exception as e:
        LAST_ERROR_MESSAGE = str(e)
        print("Mistral API error:", e)
        return None

# 4) GLADIA VOICE

In [6]:
def transcribe_audio_with_gladia(file_path: str) -> Optional[str]:
    if not GLADIA_API_KEY:
        print("Gladia API key is not set.")
        return None

    headers = {
        "x-gladia-key": GLADIA_API_KEY,
    }

    try:
        with open(file_path, "rb") as f:
            files = {"audio": (file_path, f, "audio/wav")}
            upload_resp = requests.post(
                f"{GLADIA_BASE_URL}/v2/upload",
                headers=headers,
                files=files,
                timeout=60,
            )
        upload_resp.raise_for_status()
        upload_json = upload_resp.json()
        audio_url = upload_json.get("audio_url")
        if not audio_url:
            print("Gladia upload did not return audio_url:", upload_json)
            return None

        pre_recorded_headers = {
            **headers,
            "Content-Type": "application/json",
        }
        data = {
            "audio_url": audio_url,
        }
        init_resp = requests.post(
            f"{GLADIA_BASE_URL}/v2/pre-recorded",
            headers=pre_recorded_headers,
            json=data,
            timeout=60,
        )
        init_resp.raise_for_status()
        init_json = init_resp.json()
        job_id = init_json.get("id")
        if not job_id:
            print("Gladia pre-recorded init did not return id:", init_json)
            return None

        for _ in range(20):
            time.sleep(3)
            result_resp = requests.get(
                f"{GLADIA_BASE_URL}/v2/pre-recorded/{job_id}",
                headers=headers,
                timeout=60,
            )
            result_resp.raise_for_status()
            result_json = result_resp.json()
            status = result_json.get("status")
            if status == "done":
                result = result_json.get("result") or {}

                transcription = None
                if isinstance(result, dict):
                    transcription_obj = result.get("transcription")
                    if isinstance(transcription_obj, dict):
                        transcription = transcription_obj.get("full_transcript")

                    if not transcription and "results" in result:
                        segments = result.get("results") or []
                        transcription = " ".join(seg.get("text", "") for seg in segments)

                return transcription

            if status == "error":
                print("Gladia transcription error:", result_json.get("error_code"))
                return None

        print("Gladia transcription timed out.")
        return None

    except Exception as e:
        print("Error while calling Gladia:", e)
        return None

# 5) PATIENT SIDE

In [7]:
def triage_app(name, symptoms, temperature_c, blood_pressure, heart_rate_bpm):
    global PATIENT_REQUESTS

    if not symptoms.strip():
        return ("Please describe your symptoms.", "", "", "", "", "")

    llm_input_obj = {
        "name": name or "Unknown",
        "symptoms": symptoms,
        "temperature_c": temperature_c,
        "blood_pressure": blood_pressure,
        "heart_rate_bpm": heart_rate_bpm,
    }
    llm_input_str = json.dumps(llm_input_obj, ensure_ascii=False)
    print("PROMPT SENT TO MODEL:\n", llm_input_str)

    debug_prompt = f"LLM input sent to model:\n```json\n{llm_input_str}\n```"

    pre_diag = make_prediagnosis(llm_input_str)

    patient = PatientRequest(
        name=name or "Unknown",
        symptoms=symptoms,
        temperature_c=temperature_c,
        blood_pressure=blood_pressure,
        heart_rate_bpm=heart_rate_bpm,
        pre_diagnosis=pre_diag,
    )
    PATIENT_REQUESTS.append(patient)

    if pre_diag is None:
        msg = (
            "Automatic pre-diagnosis failed.\n"
            "A human operator must review this case.\n\n"
            f"Error details: `{LAST_ERROR_MESSAGE}`"
        )
        return (
            msg,
            f"LLM input:\n```json\n{llm_input_str}\n```",
            "",
            "",
            "",
            debug_prompt,
        )

    urgency_label = {
        "low": "🟢 Low",
        "medium": "🟠 Medium",
        "high": "🔴 High"
    }.get(pre_diag.urgencyLevel, "Unknown")

    summary = f"""
**Patient:** {patient.name}
**Urgency:** {urgency_label}
**Suspected condition:** {pre_diag.condition}
"""

    symptoms_out = pre_diag.symptoms
    vitals_out = (
        f"- Temperature: {patient.temperature_c} °C\n"
        f"- Blood pressure: {patient.blood_pressure}\n"
        f"- Heart rate: {patient.heart_rate_bpm} bpm\n"
    )

    json_out = json.dumps({
        "name": patient.name,
        "preDiagnosis": {
            "condition": pre_diag.condition,
            "urgencyLevel": pre_diag.urgencyLevel,
            "symptoms": pre_diag.symptoms,
        },
        "vitals": {
            "temperature": patient.temperature_c,
            "blood_pressure": patient.blood_pressure,
            "heart_rate": patient.heart_rate_bpm,
        }
    }, indent=2, ensure_ascii=False)

    return (summary, symptoms_out, vitals_out, pre_diag.urgencyLevel, json_out, debug_prompt)


def voice_triage(audio_file, name, temperature_c, blood_pressure, heart_rate_bpm):
    """
    New: voice path – patient records symptoms, we transcribe with Gladia,
    then reuse triage_app with the transcript as symptoms.
    """
    if audio_file is None:
        msg = "Please record your symptoms using the microphone."
        return (msg, "", "", "", "", "No audio recorded.")

    transcript = transcribe_audio_with_gladia(audio_file)
    if not transcript:
        msg = (
            "Voice transcription failed.\n"
            "Please try again or type your symptoms manually."
        )
        return (msg, "", "", "", "", "Gladia transcription failed.")

    summary, symptoms_out, vitals_out, urgency, json_out, prompt = triage_app(
        name,
        transcript,
        temperature_c,
        blood_pressure,
        heart_rate_bpm,
    )

    combined_symptoms_out = (
        f"Transcribed symptoms from voice:\n{transcript}\n\n"
        "---\n\n"
        f"{symptoms_out}"
    )

    return (summary, combined_symptoms_out, vitals_out, urgency, json_out, prompt)


def voice_to_symptoms(audio_file, current_symptoms: str):
    """
    Record audio, transcribe with Gladia, and return text to put into the
    'Describe your symptoms' field. Does NOT run analysis.
    """
    if audio_file is None:
        return current_symptoms or ""

    transcript = transcribe_audio_with_gladia(audio_file)
    if not transcript:
        return current_symptoms or ""

    return transcript

# 6) STAFF DASHBOARD

In [8]:
def load_staff_table(filter_urgency):
    rows = []
    for idx, req in enumerate(PATIENT_REQUESTS, start=1):
        if req.pre_diagnosis:
            urg = req.pre_diagnosis.urgencyLevel
            cond = req.pre_diagnosis.condition
        else:
            urg = "n/a"
            cond = "n/a"

        if filter_urgency != "All" and urg != filter_urgency:
            continue

        rows.append([
            idx,
            req.name,
            cond,
            urg,
            req.temperature_c,
            req.blood_pressure,
            req.heart_rate_bpm,
            req.symptoms
        ])

    return rows

# 7) GRADIO APP

In [9]:

with gr.Blocks() as demo:
    gr.Markdown("SAMU – AI Pre-Triage System")

    with gr.Tabs():
# TAB 1: Patient
        with gr.Tab("​Patient interface"):
            gr.Markdown("Submit symptoms + vitals to get a pre-diagnosis.")

            name = gr.Textbox(label="Patient name")
            symptoms = gr.Textbox(label="Describe your symptoms", lines=5)
            temperature_c = gr.Number(label="Temperature (°C)", value=37.0)
            blood_pressure = gr.Textbox(label="Blood pressure (mmHg)", value="120/80")
            heart_rate = gr.Number(label="Heart rate (bpm)", value=70)

            audio_input = gr.Audio(
              type="filepath",
              label="Record your symptoms (voice)",
              sources=["microphone"]
            )
            run_voice_btn = gr.Button("Record & transcribe voice to symptoms")
            spacer = gr.HTML("<div style='height: 16px;'></div>")
            run_btn = gr.Button("Analyze symptoms")

            summary_out = gr.Markdown()
            symptoms_out = gr.Markdown()
            vitals_out = gr.Markdown()
            urgency_out = gr.Textbox(label="Urgency level")
            json_out = gr.Code()
            prompt_out = gr.Markdown(label="Prompt sent to model")

            run_btn.click(
                triage_app,
                inputs=[name, symptoms, temperature_c, blood_pressure, heart_rate],
                outputs=[summary_out, symptoms_out, vitals_out, urgency_out, json_out, prompt_out]
            )

            run_voice_btn.click(
                voice_to_symptoms,
                inputs=[audio_input, symptoms],
                outputs=[symptoms],
            )

# TAB 2: Staff dashboard
        with gr.Tab("Medical staff dashboard"):
            gr.Markdown("List of all incoming patient requests.")

            urgency_filter = gr.Dropdown(
                ["All", "low", "medium", "high"],
                value="All",
                label="Filter by urgency"
            )
            refresh_btn = gr.Button("Refresh")

            table = gr.Dataframe(
                headers=[
                    "ID", "Patient", "Condition",
                    "Urgency", "Temp (°C)",
                    "BP (mmHg)", "HR (bpm)", "Symptoms"
                ],
                value=[]
            )

            refresh_btn.click(load_staff_table, inputs=urgency_filter, outputs=table)

demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
